In [13]:
import pandas as pd
import numpy as np
import os

In [14]:
depth_values = range(1,11,2)
criteria = ["gini", "entropy"]
leaves = [1,2,5,10]
splits = [1,2,5,10]

datasets = ["bc", "hp", "hpimp"]
hyperparams = ["max_depth", "criterion", "min_samples_leaf", "min_samples_split"]
kinds = ["train", "test"]
columns = ["dataset", "hyperparam", "value", "kind", "accuracy", "precision", "sensitivity", "specificity", "f1"]
metrics_df = pd.DataFrame(columns=columns)

In [15]:
def get_metrics(df):
    y_true = df["y_test"].values
    y_pred = df["y_pred"].values

    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0 
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    false_positive_rate = FP / (FP + TN) if (FP + TN) > 0 else 0
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "f1": f1
    }

    return metrics

In [16]:
folder = "../predictions_dt/"
for file in os.listdir(folder):
    if not file.endswith(".csv"):
        continue

    df = pd.read_csv(os.path.join(folder, file))
    
    # parse filename: bc_predictions_max_depth_3_train.csv
    name = file.replace(".csv","").split("_")
    dataset = name[0]      
    hyperparam = "_".join(name[2:-2])             
    value = name[-2]  
    kind = name[-1]                  

    # convert numeric values if possible
    try:
        value = int(value)
    except ValueError:
        pass  # keep string like "gini"/"entropy"

    # compute metrics
    metrics = get_metrics(df)

    row = {
        "dataset": dataset,
        "hyperparam": hyperparam,
        "value": value,
        "kind": kind,
        **metrics
    }
    metrics_df.loc[len(metrics_df)] = row

metrics_df

,dataset,hyperparam,value,kind,accuracy,precision,sensitivity,specificity,f1
0,bc,criterion,entropy,test,0.946903,1.00000,0.869565,1.000000,0.930233
1,bc,criterion,entropy,train,0.978070,1.00000,0.939759,1.000000,0.968944
2,bc,criterion,gini,test,0.938053,0.97561,0.869565,0.985075,0.919540
3,bc,criterion,gini,train,0.969298,1.00000,0.915663,1.000000,0.955975
4,bc,max_depth,10,test,0.964602,0.93750,0.978261,0.955224,0.957447
...,...,...,...,...,...,...,...,...,...
115,hp,min_samples_split,1,train,0.968750,1.00000,0.818182,1.000000,0.900000
116,hp,min_samples_split,2,test,0.687500,0.20000,0.500000,0.714286,0.285714
117,hp,min_samples_split,2,train,0.968750,1.00000,0.818182,1.000000,0.900000
118,hp,min_samples_split,5,test,0.687500,0.20000,0.500000,0.714286,0.285714


In [17]:
metrics_df.to_csv("dt_metrics.csv", index=False)